# 01 – Data Cleaning & Preparation
**Goal:** understand the three Walmart tables, check quality, merge them, and save one clean dataset for EDA and modelling.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

train = pd.read_csv(RAW / "train.csv", parse_dates=["Date"])
features = pd.read_csv(RAW / "features.csv", parse_dates=["Date"])
stores = pd.read_csv(RAW / "stores.csv")

for name, df in [("train", train), ("features", features), ("stores", stores)]:
    print(name, df.shape)
    display(df.head(3))

train (421570, 5)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False


features (8190, 12)


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False


stores (45, 3)


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392


In [2]:
print(train.describe())
print(train.info())

               Store           Dept                        Date   Weekly_Sales
count  421570.000000  421570.000000                      421570  421570.000000
mean       22.200546      44.260317  2011-06-18 08:30:31.963375   15981.258123
min         1.000000       1.000000         2010-02-05 00:00:00   -4988.940000
25%        11.000000      18.000000         2010-10-08 00:00:00    2079.650000
50%        22.000000      37.000000         2011-06-17 00:00:00    7612.030000
75%        33.000000      74.000000         2012-02-24 00:00:00   20205.852500
max        45.000000      99.000000         2012-10-26 00:00:00  693099.360000
std        12.785297      30.492054                         NaN   22711.183519
<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  i

In [3]:
print(features.describe())
print(features.info())

             Store                 Date  Temperature   Fuel_Price  \
count  8190.000000                 8190  8190.000000  8190.000000   
mean     23.000000  2011-10-31 12:00:00    59.356198     3.405992   
min       1.000000  2010-02-05 00:00:00    -7.290000     2.472000   
25%      12.000000  2010-12-17 00:00:00    45.902500     3.041000   
50%      23.000000  2011-10-31 12:00:00    60.710000     3.513000   
75%      34.000000  2012-09-14 00:00:00    73.880000     3.743000   
max      45.000000  2013-07-26 00:00:00   101.950000     4.468000   
std      12.987966                  NaN    18.678607     0.431337   

           MarkDown1      MarkDown2      MarkDown3     MarkDown4  \
count    4032.000000    2921.000000    3613.000000   3464.000000   
mean     7032.371786    3384.176594    1760.100180   3292.935886   
min     -2781.450000    -265.760000    -179.260000      0.220000   
25%      1577.532500      68.880000       6.600000    304.687500   
50%      4743.580000     364.570000   

In [4]:
print(stores.describe())
print(stores.info())

           Store           Size
count  45.000000      45.000000
mean   23.000000  130287.600000
std    13.133926   63825.271991
min     1.000000   34875.000000
25%    12.000000   70713.000000
50%    23.000000  126512.000000
75%    34.000000  202307.000000
max    45.000000  219622.000000
<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Store   45 non-null     int64
 1   Type    45 non-null     str  
 2   Size    45 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.2 KB
None


## 1. Data Understanding
### 1.1 Time span, stores, departments

In [5]:
print("Train date range: ", train["Date"].min().date(), "-->", train["Date"].max().date())
print("Features date range: ", features["Date"].min().date(), "-->", features["Date"].max().date())
print("Stores: ", train["Store"].nunique(), "|Departments: ", train["Dept"].nunique())
print("Unique weeks in train: ", train["Date"].nunique())
print("Store-Dept time series: ", train.groupby(["Store", "Dept"]).ngroups)

# Are weeks evenly spaces?
gaps = train["Date"].drop_duplicates().sort_values().diff().dropna()
print("\nGap between weeks (days):")
print(gaps.dt.days.value_counts())

# Weeks flagged as holidays
print("\nHoliday weeks: ", sorted(train.loc[train["IsHoliday"], "Date"].dt.date.unique()))

Train date range:  2010-02-05 --> 2012-10-26
Features date range:  2010-02-05 --> 2013-07-26


Stores:  45 |Departments:  81
Unique weeks in train:  143
Store-Dept time series:  3331

Gap between weeks (days):
Date
7    142
Name: count, dtype: int64

Holiday weeks:  [datetime.date(2010, 2, 12), datetime.date(2010, 9, 10), datetime.date(2010, 11, 26), datetime.date(2010, 12, 31), datetime.date(2011, 2, 11), datetime.date(2011, 9, 9), datetime.date(2011, 11, 25), datetime.date(2011, 12, 30), datetime.date(2012, 2, 10), datetime.date(2012, 9, 7)]


**Observations**
- Train covers 2010-02-05 to 2012-10-26 (143 weekly observations, evenly spaced at 7 days). `features.csv` extends to 2013-07-26, which is the unlabelled Kaggle test period, so it is not used for modelling.
- 45 stores, 81 departments and 3,331 store–department series.
- Only 10 holiday weeks appear in train (Super Bowl, Labor Day, Thanksgiving and the week of Dec 30/31 in 2010–2011; only Super Bowl and Labor Day in 2012).
- The holiday flag for Christmas falls on the week *after* Dec 25, not the pre-Christmas peak week (see 3.4).

### 1.2 Missing Values

In [6]:
def missing_report(df):
    m=df.isna().sum()
    out=pd.DataFrame({"missing": m, "missing_pct": (m/len(df)*100).round(2)})
    return out[out['missing']>0]

print("TRAIN:\n", missing_report(train), "\n")
print("STORES:\n", missing_report(stores), "\n")
print("Features:\n", missing_report(features), "\n")

# Only look at the period that overlaps with train
feat_train_period = features[features["Date"] <= train["Date"].max()]
print("FEATURES (train period only):\n", missing_report(feat_train_period))

first_md = features.loc[features["MarkDown1"].notna(), "Date"].min()
print("\nFirst date with markdown data:", first_md.date())

TRAIN:
 Empty DataFrame
Columns: [missing, missing_pct]
Index: [] 

STORES:
 Empty DataFrame
Columns: [missing, missing_pct]
Index: [] 

Features:
               missing  missing_pct
MarkDown1        4158        50.77
MarkDown2        5269        64.33
MarkDown3        4577        55.89
MarkDown4        4726        57.70
MarkDown5        4140        50.55
CPI               585         7.14
Unemployment      585         7.14 

FEATURES (train period only):
            missing  missing_pct
MarkDown1     4155        64.57
MarkDown2     4798        74.56
MarkDown3     4389        68.21
MarkDown4     4470        69.46
MarkDown5     4140        64.34

First date with markdown data: 2011-11-11


**Observations**
- `train` and `stores` have no missing values.
- `CPI` and `Unemployment` are missing only after the train period (future/test weeks), so they are complete wherever we have sales.
- `MarkDown1–5` are missing for roughly 64–75% of train-period rows. Markdown data first appears on 2011-11-11, so most of this is structural (not tracked yet), although some gaps also occur after tracking began. This is missing by design, not random error.

### 1.3 Duplicates

In [7]:
print("Duplicate Store-Dept-Date rows in train:", train.duplicated(subset=["Store", "Dept", "Date"]).sum())
print("Duplicate Store-Date rows in features:  ", features.duplicated(subset=["Store", "Date"]).sum())
print("Duplicate Store rows in stores:         ", stores["Store"].duplicated().sum())
print("Exact duplicate rows in train:          ", train.duplicated().sum())

Duplicate Store-Dept-Date rows in train: 0
Duplicate Store-Date rows in features:   0
Duplicate Store rows in stores:          0
Exact duplicate rows in train:           0


### 1.4 Is `IsHoliday` consistent between train and features?
We plan to merge on Store, Date and IsHoliday, so confirm they agree.

In [8]:
chk = train[["Store", "Date", "IsHoliday"]].drop_duplicates().merge(
    features[["Store", "Date", "IsHoliday"]],
    on=["Store", "Date"], how="left", suffixes=("_train", "_feat"))
print("Mismatching IsHoliday flags:", (chk["IsHoliday_train"] != chk["IsHoliday_feat"]).sum())
print("Train rows without a features match:", chk["IsHoliday_feat"].isna().sum())

Mismatching IsHoliday flags: 0
Train rows without a features match: 0


**Result:** no duplicate keys anywhere, and `IsHoliday` is identical in `train` and `features` for every store-week, so it is safe to drop it from `features` before merging.

## 2. Merging the tables
`train` (weekly sales) + `stores` (Type, Size) on **Store**, then + `features` (weather, fuel, markdowns, CPI, unemployment) on **Store, Date**.
We use a left join so no sales rows are lost, and `validate` to guard against accidental row duplication.

In [9]:
df = (
    train
    .merge(stores, on="Store", how="left", validate="m:1")
    .merge(features.drop(columns="IsHoliday"), on=["Store", "Date"], how="left", validate="m:1")
)
print("Rows before: ",  len(train), "after: ", len(df))
assert len(train) == len(df), "Row count mismatch after merge"
print("missing Type/Size after merge: ", df[["Type", "Size"]].isna().sum().sum())
print(f"Missing values after merge:\n{missing_report(df)}")
df.head()

Rows before:  421570 after:  421570
missing Type/Size after merge:  0
Missing values after merge:
           missing  missing_pct
MarkDown1   270889        64.26
MarkDown2   310322        73.61
MarkDown3   284479        67.48
MarkDown4   286603        67.98
MarkDown5   270138        64.08


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106


## 3. Cleaning
### 3.1 Negative and zero sales
Negative `Weekly_Sales` most likely represent returns/adjustments. Before deciding what to do, measure how common they are.

In [10]:
neg = df[df["Weekly_Sales"] < 0]
zero = df[df["Weekly_Sales"] == 0]
print(f"Negative sales rows: {len(neg)} ({len(neg)/len(df)*100:.3f}%)")
print(f"Zero sales rows:     {len(zero)} ({len(zero)/len(df)*100:.3f}%)")
print("\nDepartments with most negative rows:")
print(neg["Dept"].value_counts().head(5))

Negative sales rows: 1285 (0.305%)
Zero sales rows:     73 (0.017%)

Departments with most negative rows:
Dept
47    254
18    180
54    146
19     87
94     77
Name: count, dtype: int64


**Decision:** negative sales are 1,285 rows (0.305%) and zero sales 73 rows (0.017%). They concentrate in a few departments (47, 18, 54), consistent with returns or adjustments. We **keep them** as real data and add a flag `neg_sales`. If they hurt model training, we will revisit this in the modelling notebook and document the change.

In [11]:
df["neg_sales"] = (df["Weekly_Sales"] < 0).astype(int)

### 3.2 Markdown columns
Missing markdowns mean "no promotion data recorded" (before Nov 2011), not unknown values. We fill NaN with 0 and add a flag `has_markdown_data` so models can tell "no promo" from "not tracked yet".

In [12]:
md_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

df["has_markdown_data"] = df[md_cols].notna().any(axis=1).astype(int)
print("Negative markdown values:", (df[md_cols] < 0).sum().to_dict())

df[md_cols] = df[md_cols].fillna(0)
print("Remaining missing in markdowns:", df[md_cols].isna().sum().sum())

Negative markdown values: {'MarkDown1': 0, 'MarkDown2': 1311, 'MarkDown3': 257, 'MarkDown4': 0, 'MarkDown5': 0}
Remaining missing in markdowns: 0


In [18]:
post = df[df["Date"] >= first_md]
print("Post-Nov-2011 rows with no markdown at all:", (post["has_markdown_data"] == 0).sum())

Post-Nov-2011 rows with no markdown at all: 0


**Result:** no remaining gaps in CPI, Unemployment, Temperature or Fuel_Price for the train period, so no imputation is needed.

**Check:** every row on or after 2011-11-11 has at least one markdown value, so `has_markdown_data` correctly marks the period when markdown tracking was active. Remaining gaps within that period are individual markdown types not running that week, so filling them with 0 is appropriate.

**Note:** MarkDown2 (1,311 rows) and MarkDown3 (257 rows) contain negative values, likely adjustments or corrections. They are kept as-is; tree-based models are not sensitive to this, and it will be revisited if needed.

### 3.3 CPI and Unemployment
Check whether any gaps remain in the train period.

In [13]:
print(df[["CPI", "Unemployment", "Temperature", "Fuel_Price"]].isna().sum())

CPI             0
Unemployment    0
Temperature     0
Fuel_Price      0
dtype: int64


### 3.4 Outliers
The largest values are not errors. The top weekly sales are almost all **Dept 72 in Thanksgiving week** (Stores 10, 35, 14, 20, 27), plus two in the week of **2010-12-24**.

**Important:** 2010-12-24 is flagged `IsHoliday = False`, so the dataset's holiday flag does not capture the main pre-Christmas sales peak. Holiday-week averages are only modestly higher than non-holiday (about 17,036 vs 15,901) because the flag also covers weaker holidays such as Super Bowl and Labor Day. We keep all outliers and may add our own holiday features (e.g. week of Thanksgiving, pre-Christmas) in feature engineering.

In [14]:
cols = ["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday", "Type"]
print(df.nlargest(10, "Weekly_Sales")[cols])

print("\nAverage weekly sales: holiday vs non-holiday")
print(df.groupby("IsHoliday")["Weekly_Sales"].agg(["mean", "median", "count"]))

        Store  Dept       Date  Weekly_Sales  IsHoliday Type
95373      10    72 2010-11-26     693099.36       True    B
338013     35    72 2011-11-25     649770.18       True    B
95425      10    72 2011-11-25     630999.19       True    B
337961     35    72 2010-11-26     627962.93       True    B
135665     14    72 2010-11-26     474330.10       True    A
195088     20    72 2010-11-26     422306.25       True    A
264390     27    72 2010-11-26     420586.57       True    A
88428      10     7 2010-12-24     406988.63      False    B
95377      10    72 2010-12-24     404245.03      False    B
214432     22    72 2010-11-26     393705.20       True    B

Average weekly sales: holiday vs non-holiday
                   mean   median   count
IsHoliday                               
False      15901.445069  7589.95  391909
True       17035.823187  7947.74   29661


### 3.5 Series length
Not every Store–Department combination has all 143 weeks. This affects lag features and per-series models.

In [15]:
series_len = df.groupby(["Store", "Dept"]).size()
print(series_len.describe())
print("Series with full history (143 weeks):", (series_len == series_len.max()).sum(), "of", len(series_len))
print("Series with fewer than 52 weeks:     ", (series_len < 52).sum())

count    3331.000000
mean      126.559592
std        40.212763
min         1.000000
25%       143.000000
50%       143.000000
75%       143.000000
max       143.000000
dtype: float64
Series with full history (143 weeks): 2660 of 3331
Series with fewer than 52 weeks:      340


**Result:** 2,660 of 3,331 series (about 80%) have the full 143 weeks, but 340 series have fewer than 52 weeks (the shortest has just 1). For those, `lag_52` and a seasonal-naive forecast are not possible. We will need a clear rule in modelling, such as a fallback forecast or excluding very short series, and will document it in the report.

## 4. Final checks and saving

In [16]:
df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

# Type tidying
df["Type"] = df["Type"].astype("category")
df["IsHoliday"] = df["IsHoliday"].astype(int)

print("Shape:", df.shape)
print("Any missing values left:", df.isna().sum().sum())
df.info()

Shape: (421570, 18)
Any missing values left: 0
<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Store              421570 non-null  int64         
 1   Dept               421570 non-null  int64         
 2   Date               421570 non-null  datetime64[us]
 3   Weekly_Sales       421570 non-null  float64       
 4   IsHoliday          421570 non-null  int64         
 5   Type               421570 non-null  category      
 6   Size               421570 non-null  int64         
 7   Temperature        421570 non-null  float64       
 8   Fuel_Price         421570 non-null  float64       
 9   MarkDown1          421570 non-null  float64       
 10  MarkDown2          421570 non-null  float64       
 11  MarkDown3          421570 non-null  float64       
 12  MarkDown4          421570 non-null  float64       
 13  MarkDown

In [17]:
from pathlib import Path
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

df.to_parquet(OUT / "merged_clean.parquet", index=False)
df.to_csv(OUT / "merged_clean.csv", index=False)   # readable copy
print("Saved.")

Saved.


## 5. Summary of findings
- Dataset: 421,570 weekly records, 45 stores, 81 departments, 3,331 store–dept series, Feb 2010 – Oct 2012.
- No duplicates; `IsHoliday` consistent across tables; future-period features (to Jul 2013) are unused.
- MarkDowns start 2011-11-11 → filled with 0 + `has_markdown_data` flag; negative markdowns kept.
- Negative sales are rare (0.305%) → kept and flagged (`neg_sales`).
- The holiday flag misses the pre-Christmas peak week (2010-12-24) → consider custom holiday features.
- About 20% of series are incomplete, 340 have under 52 weeks → affects lag features and the seasonal-naive baseline.
- Output: `data/processed/merged_clean.parquet` (18 columns, no missing values).

**Next:** `02_eda.ipynb`